# CAMS Date-Range Downloader

This notebook downloads CAMS leadtime-0 data between `START_DATE` and `END_DATE` into `/mnt/data3/cams`.

Requirements:
- `pip install cdsapi`
- `~/.cdsapirc` points to `https://ads.atmosphere.copernicus.eu/api`
- CAMS dataset terms accepted in ADS


In [1]:
from datetime import datetime
from pathlib import Path

START_DATE = "2025-04-16" # YYYY-MM-DD
END_DATE = "2025-04-30"    # YYYY-MM-DD
OUTPUT_DIR = Path("/mnt/data3/cams")
TIMES_UTC = ["00:00", "12:00"]
OVERWRITE = False
REMOVE_ZIP_AFTER_EXTRACTION = True

start_dt = datetime.strptime(START_DATE, "%Y-%m-%d")
end_dt = datetime.strptime(END_DATE, "%Y-%m-%d")
if end_dt < start_dt:
    raise ValueError("END_DATE must be on or after START_DATE")

date_range = START_DATE if START_DATE == END_DATE else f"{START_DATE}/{END_DATE}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

base_name = f"{START_DATE}_to_{END_DATE}-cams-range-lead0"
zip_path = OUTPUT_DIR / f"{base_name}.nc.zip"
surface_path = OUTPUT_DIR / f"{base_name}-surface-level.nc"
atmos_path = OUTPUT_DIR / f"{base_name}-atmospheric.nc"

print(f"Date range: {date_range}")
print(f"Download zip: {zip_path}")
print(f"Surface file: {surface_path}")
print(f"Atmos file: {atmos_path}")


Date range: 2025-04-16/2025-04-30
Download zip: /mnt/data3/cams/2025-04-16_to_2025-04-30-cams-range-lead0.nc.zip
Surface file: /mnt/data3/cams/2025-04-16_to_2025-04-30-cams-range-lead0-surface-level.nc
Atmos file: /mnt/data3/cams/2025-04-16_to_2025-04-30-cams-range-lead0-atmospheric.nc


In [2]:
CAMS_VARIABLES = [
    # Meteorological surface-level variables
    "10m_u_component_of_wind",
    "10m_v_component_of_wind",
    "2m_temperature",
    "mean_sea_level_pressure",
    # Pollution surface-level variables
    "particulate_matter_1um",
    "particulate_matter_2.5um",
    "particulate_matter_10um",
    "total_column_carbon_monoxide",
    "total_column_nitrogen_monoxide",
    "total_column_nitrogen_dioxide",
    "total_column_ozone",
    "total_column_sulphur_dioxide",
    # Meteorological atmospheric variables
    "u_component_of_wind",
    "v_component_of_wind",
    "temperature",
    "geopotential",
    "specific_humidity",
    # Pollution atmospheric variables
    "carbon_monoxide",
    "nitrogen_dioxide",
    "nitrogen_monoxide",
    "ozone",
    "sulphur_dioxide",
]

CAMS_PRESSURE_LEVELS = [
    "50", "100", "150", "200", "250", "300",
    "400", "500", "600", "700", "850", "925", "1000",
]


In [3]:
import zipfile

import cdsapi


def _exists_nonempty(path: Path) -> bool:
    return path.exists() and path.stat().st_size > 0


request = {
    "type": "forecast",
    "leadtime_hour": "0",
    "variable": CAMS_VARIABLES,
    "pressure_level": CAMS_PRESSURE_LEVELS,
    "date": date_range,
    "time": TIMES_UTC,
    "format": "netcdf_zip",
}

if _exists_nonempty(zip_path) and not OVERWRITE:
    print(f"Skipping existing zip: {zip_path}")
else:
    client = cdsapi.Client()
    client.retrieve(
        "cams-global-atmospheric-composition-forecasts",
        request,
        str(zip_path),
    )
    print(f"Downloaded: {zip_path}")

if _exists_nonempty(surface_path) and _exists_nonempty(atmos_path) and not OVERWRITE:
    print("Skipping extraction because output NetCDF files already exist.")
else:
    with zipfile.ZipFile(zip_path, "r") as zf:
        with open(surface_path, "wb") as f:
            f.write(zf.read("data_sfc.nc"))
        with open(atmos_path, "wb") as f:
            f.write(zf.read("data_plev.nc"))
    print(f"Extracted: {surface_path}")
    print(f"Extracted: {atmos_path}")

if REMOVE_ZIP_AFTER_EXTRACTION and zip_path.exists() and _exists_nonempty(surface_path) and _exists_nonempty(atmos_path):
    zip_path.unlink()
    print(f"Removed zip: {zip_path}")


2026-05-01 20:05:37,690 INFO Request ID is 104f13da-6f61-4396-8508-9bb93b52b5dd
2026-05-01 20:05:37,902 INFO status has been updated to accepted
2026-05-01 20:05:52,191 INFO status has been updated to running
2026-05-01 20:12:00,336 INFO status has been updated to failed


HTTPError: 400 Client Error: Bad Request for url: https://ads.atmosphere.copernicus.eu/api/retrieve/v1/jobs/104f13da-6f61-4396-8508-9bb93b52b5dd/results
The job has failed
MARS has returned an error, please check your selection.
Request submitted to the MARS server:
[{'date': ['2025-04-16', '2025-04-17', '2025-04-18', '2025-04-19', '2025-04-20', '2025-04-21', '2025-04-22', '2025-04-23', '2025-04-24', '2025-04-25', '2025-04-26', '2025-04-27', '2025-04-28', '2025-04-29', '2025-04-30'], 'grid': ['0.4', '0.4'], 'step': ['0'], 'levelist': ['50', '100', '150', '200', '250', '300', '400', '500', '600', '700', '850', '925', '1000'], 'time': ['12:00:00'], 'type': ['fc'], 'param': ['131', '132', '130', '129', '133', '210123', '210121', '217027', '210203', '210122'], 'class': ['mc'], 'expect': ['off'], 'expver': ['0001'], 'number': ['all'], 'stream': ['oper'], 'levtype': ['pl'], 'database': ['cdsfdb/external']}, {'date': ['2025-04-16', '2025-04-17', '2025-04-18', '2025-04-19', '2025-04-20', '2025